# ToM Parameter Identification - Original Paper Methodology

This notebook implements the methodology from "How large language models encode theory-of-mind: a study on sparse parameter patterns."

## Overview

The pipeline identifies ToM-sensitive parameters by:
1. Computing Fisher Information Matrix (FIM) approximations via squared gradients on ToM dataset
2. Computing FIM on C4 (generic text) dataset
3. Identifying parameters with high ToM gradient but low C4 gradient
4. Perturbing these parameters and measuring impact on ToM tasks vs perplexity

**Epistemic Status**: High confidence in implementation fidelity to paper (code is from paper authors). However, note methodological concerns about token supervision imbalance (C4 has ~128× more supervised tokens per sample).

In [ ]:
# Setup and imports
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess

# Configuration - MODIFY THESE AS NEEDED
BASE_MODEL = "meta-llama/Llama-3.2-1B"  # Small LLaMA variant for testing
CACHE_DIR = "./cache"
PERMANENT_STORAGE = "./permanent_storage"
TMP_DIR = "./tmp"

# Gradient computation settings
TOM_NSAMPLES = 100  # Number of ToM samples for gradient computation
C4_NSAMPLES = 100   # Number of C4 samples for gradient computation
C4_SEQLEN = 128     # C4 sequence length
TOM_SEQLEN = 0      # 0 = use full sequence

# Evaluation settings
M_VALUES = [0.0, 1e-5, 2e-5, 3e-5, 4e-5, 5e-5]  # Sparsity levels to test
EVAL_REPS = 5       # Number of repetitions for ToM evaluation
BATCH_SIZE = 64     # vLLM batch size
MAX_MODEL_LEN = 1024  # vLLM max sequence length
TENSOR_PARALLEL_SIZE = 1  # GPU parallelism (set to 1 for single GPU)

# Dataset paths
TOM_TRAINING_DATA = "./tom_training_data.json"
TOM_TASKS_FILE = "./ToM_tasks.py"

# Create directories
os.makedirs(PERMANENT_STORAGE, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  Model: {BASE_MODEL}")
print(f"  ToM samples: {TOM_NSAMPLES}, C4 samples: {C4_NSAMPLES}")
print(f"  M values: {M_VALUES}")

## Step 1: Compute Gradients (Fisher Information)

**Epistemic Status**: High confidence. Uses existing `create_gradient.py` which:
- Registers hooks to square gradients: g → g²
- For ToM: supervises only the last token per sample (~100 tokens total)
- For C4: supervises all 128 tokens per sample (~12,800 tokens total)

**Important**: This creates ~128× token supervision imbalance. The paper does not normalize for this.

**Note**: Requires GPU and HuggingFace access. Expected runtime: 30min-3hrs depending on model size and nsamples.

In [ ]:
# Define gradient checkpoint paths (PERMANENT storage)
TOM_GRAD_CHECKPOINT = os.path.join(PERMANENT_STORAGE, "gradients", "tom_grad")
C4_GRAD_CHECKPOINT = os.path.join(PERMANENT_STORAGE, "gradients", "c4_grad")

os.makedirs(os.path.dirname(TOM_GRAD_CHECKPOINT), exist_ok=True)
os.makedirs(os.path.dirname(C4_GRAD_CHECKPOINT), exist_ok=True)

print(f"Gradient checkpoints will be saved to:")
print(f"  ToM: {TOM_GRAD_CHECKPOINT}")
print(f"  C4:  {C4_GRAD_CHECKPOINT}")

In [ ]:
# Unit test: Verify dataset exists
assert os.path.exists(TOM_TRAINING_DATA), f"Missing ToM training data at {TOM_TRAINING_DATA}"

# Verify dataset format
with open(TOM_TRAINING_DATA, 'r') as f:
    tom_data = json.load(f)
    assert isinstance(tom_data, list), "ToM data should be a list"
    assert len(tom_data) > 0, "ToM data is empty"
    assert 'txt' in tom_data[0], "ToM data entries should have 'txt' key"
    print(f"✓ ToM dataset validated: {len(tom_data)} samples")
    print(f"  Sample 0 length: {len(tom_data[0]['txt'])} chars")

In [ ]:
# Compute ToM gradients
# EPISTEMIC STATUS: High confidence in code, but requires GPU - will fail without it

def compute_tom_gradients():
    """Compute squared gradients on ToM dataset."""
    if os.path.exists(os.path.join(TOM_GRAD_CHECKPOINT, "config.json")):
        print(f"✓ ToM gradients already exist at {TOM_GRAD_CHECKPOINT}")
        return
    
    cmd = [
        "python", "create_gradient.py",
        "--model", BASE_MODEL,
        "--dataset", "tom",
        "--data_path", TOM_TRAINING_DATA,
        "--nsamples", str(TOM_NSAMPLES),
        "--seqlen", str(TOM_SEQLEN),
        "--cache_dir", CACHE_DIR,
        "--out", TOM_GRAD_CHECKPOINT
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError("ToM gradient computation failed")
    
    print(f"✓ ToM gradients saved to {TOM_GRAD_CHECKPOINT}")
    print(result.stdout)

# Uncomment to run (requires GPU)
# compute_tom_gradients()

In [ ]:
# Compute C4 gradients
# EPISTEMIC STATUS: High confidence in code, but requires GPU - will fail without it

def compute_c4_gradients():
    """Compute squared gradients on C4 dataset."""
    if os.path.exists(os.path.join(C4_GRAD_CHECKPOINT, "config.json")):
        print(f"✓ C4 gradients already exist at {C4_GRAD_CHECKPOINT}")
        return
    
    cmd = [
        "python", "create_gradient.py",
        "--model", BASE_MODEL,
        "--dataset", "c4",
        "--nsamples", str(C4_NSAMPLES),
        "--seqlen", str(C4_SEQLEN),
        "--cache_dir", CACHE_DIR,
        "--out", C4_GRAD_CHECKPOINT
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError("C4 gradient computation failed")
    
    print(f"✓ C4 gradients saved to {C4_GRAD_CHECKPOINT}")
    print(result.stdout)

# Uncomment to run (requires GPU)
# compute_c4_gradients()

## Step 2: Chunk Gradients into Per-Layer Files

**Epistemic Status**: High confidence. This step is fast (<5min) and can regenerate from checkpoints.

Splits gradient checkpoints into `layer_{i}.pt` files for efficient masking during evaluation.

In [ ]:
# Define chunk paths (TEMPORARY storage - can regenerate)
TOM_CHUNKS_DIR = os.path.join(TMP_DIR, "chunks", "tom")
C4_CHUNKS_DIR = os.path.join(TMP_DIR, "chunks", "c4")

os.makedirs(TOM_CHUNKS_DIR, exist_ok=True)
os.makedirs(C4_CHUNKS_DIR, exist_ok=True)

print(f"Chunk directories:")
print(f"  ToM: {TOM_CHUNKS_DIR}")
print(f"  C4:  {C4_CHUNKS_DIR}")

In [ ]:
# Chunk ToM gradients
def chunk_gradients(grad_checkpoint, output_dir, name):
    """Split gradient checkpoint into per-layer chunks."""
    if os.path.exists(os.path.join(output_dir, "manifest.json")):
        print(f"✓ {name} chunks already exist at {output_dir}")
        return
    
    cmd = [
        "python", "chunk_gradient.py",
        "--model", grad_checkpoint,
        "--output_path", output_dir,
        "--cache_dir", CACHE_DIR,
        "--device_map", "cpu"  # CPU is fine for chunking
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError(f"{name} chunking failed")
    
    print(f"✓ {name} chunks saved to {output_dir}")
    print(result.stdout)

# Uncomment to run
# chunk_gradients(TOM_GRAD_CHECKPOINT, TOM_CHUNKS_DIR, "ToM")
# chunk_gradients(C4_GRAD_CHECKPOINT, C4_CHUNKS_DIR, "C4")

In [ ]:
# Unit test: Verify chunks were created correctly
def validate_chunks(chunks_dir, name):
    """Validate chunk directory structure."""
    manifest_path = os.path.join(chunks_dir, "manifest.json")
    
    if not os.path.exists(manifest_path):
        print(f"⚠ {name} chunks not yet generated")
        return False
    
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    
    num_layers = manifest['num_layers']
    
    for i in range(num_layers):
        layer_file = os.path.join(chunks_dir, f"layer_{i}.pt")
        assert os.path.exists(layer_file), f"Missing {layer_file}"
    
    print(f"✓ {name} chunks validated: {num_layers} layers")
    print(f"  Modules per layer: {manifest['modules']}")
    return True

# Uncomment to validate after chunking
# validate_chunks(TOM_CHUNKS_DIR, "ToM")
# validate_chunks(C4_CHUNKS_DIR, "C4")

## Step 3: Run Evaluation Sweep

**Epistemic Status**: High confidence in implementation. Requires GPU and vLLM.

For each m value:
1. Build masked model where mask = (ToM gradient in top-m%) AND (C4 gradient NOT in top-m%)
2. Replace masked weights with mean of non-masked weights (when scale=0.0)
3. Evaluate ToM tasks (S1, S2, S3) with vLLM
4. Evaluate perplexity on WikiText-2

**Expected runtime**: ~40min per m value for 1B model

In [ ]:
# Define evaluation output directory
EVAL_OUTPUT_DIR = os.path.join(PERMANENT_STORAGE, "evaluation_results", "paper_methodology")
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

print(f"Evaluation results will be saved to: {EVAL_OUTPUT_DIR}")

In [ ]:
# Run full evaluation sweep
# EPISTEMIC STATUS: High confidence in code, but requires GPU + vLLM

def run_evaluation_sweep():
    """Run ToM and perplexity evaluation across all m values."""
    m_list = ",".join(str(m) for m in M_VALUES)
    
    cmd = [
        "python", "ToM_and_perplexity_evaluation.py",
        "--model", BASE_MODEL,
        "--grad_tom_chunks", TOM_CHUNKS_DIR,
        "--grad_c4_chunks", C4_CHUNKS_DIR,
        "--tom_tasks", TOM_TASKS_FILE,
        "--out_dir", EVAL_OUTPUT_DIR,
        "--cache_dir", CACHE_DIR,
        "--tensor_parallel_size", str(TENSOR_PARALLEL_SIZE),
        "--max_model_len", str(MAX_MODEL_LEN),
        "--batch_size", str(BATCH_SIZE),
        "--reps", str(EVAL_REPS),
        "--m_list", m_list
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError("Evaluation sweep failed")
    
    print(f"✓ Evaluation complete")
    print(result.stdout)

# Uncomment to run (requires GPU + vLLM)
# run_evaluation_sweep()

## Step 4: Summarize Results

**Epistemic Status**: High confidence. Aggregates ToM results across repetitions.

In [ ]:
# Summarize ToM results
def summarize_tom_results():
    """Aggregate ToM results into summary CSV."""
    tom_dir = os.path.join(EVAL_OUTPUT_DIR, "tom")
    summary_csv = os.path.join(EVAL_OUTPUT_DIR, "tom_summary.csv")
    
    if not os.path.exists(tom_dir):
        print(f"⚠ ToM results not yet generated at {tom_dir}")
        return
    
    cmd = [
        "python", "summarize.py",
        "--root", tom_dir,
        "--reps", str(EVAL_REPS),
        "--out_csv", summary_csv
    ]
    
    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"ERROR: {result.stderr}")
        raise RuntimeError("Summarization failed")
    
    print(f"✓ Summary saved to {summary_csv}")
    print(result.stdout)
    
    return summary_csv

# Uncomment to run
# summary_csv = summarize_tom_results()

## Step 5: Visualize Results

**Epistemic Status**: High confidence. Standard plotting of accuracy vs m.

In [ ]:
# Plot ToM accuracy vs m
def plot_results(summary_csv, perplexity_csv):
    """Visualize ToM accuracy and perplexity vs sparsity level m."""
    
    # Load data
    if not os.path.exists(summary_csv):
        print(f"⚠ Summary CSV not found at {summary_csv}")
        return
    
    tom_df = pd.read_csv(summary_csv)
    ppl_df = pd.read_csv(perplexity_csv) if os.path.exists(perplexity_csv) else None
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot ToM accuracy
    ax1 = axes[0]
    for col in tom_df.columns:
        if col != 'm':
            ax1.plot(tom_df['m'], tom_df[col], marker='o', label=col)
    
    ax1.set_xlabel('m (sparsity level)')
    ax1.set_ylabel('ToM Accuracy')
    ax1.set_title('ToM Task Performance vs Parameter Masking')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    
    # Plot perplexity
    ax2 = axes[1]
    if ppl_df is not None:
        ax2.plot(ppl_df['m'], ppl_df['perplexity'], marker='s', color='red')
        ax2.set_xlabel('m (sparsity level)')
        ax2.set_ylabel('Perplexity')
        ax2.set_title('WikiText-2 Perplexity vs Parameter Masking')
        ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    fig_path = os.path.join(EVAL_OUTPUT_DIR, "results_visualization.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f"✓ Visualization saved to {fig_path}")
    
    plt.show()

# Uncomment to plot
# plot_results(
#     os.path.join(EVAL_OUTPUT_DIR, "tom_summary.csv"),
#     os.path.join(EVAL_OUTPUT_DIR, "perplexity_results.csv")
# )

## Complete Pipeline Runner

**Epistemic Status**: Moderate confidence. This is a convenience wrapper, but individual steps are well-tested.

**WARNING**: This will take 24-37 hours for a full experiment with a 1B model. Ensure you have:
- GPU with sufficient VRAM (≥16GB recommended)
- HuggingFace access configured
- vLLM installed
- Sufficient disk space (~12GB for 1B model)

In [ ]:
def run_complete_pipeline():
    """Run the complete ToM parameter identification pipeline."""
    print("=" * 80)
    print("STARTING COMPLETE PIPELINE")
    print("=" * 80)
    
    # Step 1: Compute gradients
    print("\n[1/5] Computing ToM gradients...")
    compute_tom_gradients()
    
    print("\n[1/5] Computing C4 gradients...")
    compute_c4_gradients()
    
    # Step 2: Chunk gradients
    print("\n[2/5] Chunking gradients...")
    chunk_gradients(TOM_GRAD_CHECKPOINT, TOM_CHUNKS_DIR, "ToM")
    chunk_gradients(C4_GRAD_CHECKPOINT, C4_CHUNKS_DIR, "C4")
    
    # Validate chunks
    validate_chunks(TOM_CHUNKS_DIR, "ToM")
    validate_chunks(C4_CHUNKS_DIR, "C4")
    
    # Step 3: Run evaluation
    print("\n[3/5] Running evaluation sweep...")
    run_evaluation_sweep()
    
    # Step 4: Summarize
    print("\n[4/5] Summarizing results...")
    summary_csv = summarize_tom_results()
    
    # Step 5: Visualize
    print("\n[5/5] Creating visualizations...")
    plot_results(
        os.path.join(EVAL_OUTPUT_DIR, "tom_summary.csv"),
        os.path.join(EVAL_OUTPUT_DIR, "perplexity_results.csv")
    )
    
    print("\n" + "=" * 80)
    print("PIPELINE COMPLETE")
    print("=" * 80)
    print(f"Results saved to: {EVAL_OUTPUT_DIR}")

# Uncomment to run complete pipeline (REQUIRES GPU + 24-37 HOURS)
# run_complete_pipeline()

## Methodological Notes

### Known Issues with Original Methodology

1. **Token Supervision Imbalance**: 
   - C4: ~12,800 supervised tokens (100 samples × 128 tokens)
   - ToM: ~100 supervised tokens (100 samples × 1 last token)
   - This creates ~128× gradient magnitude difference

2. **Dataset Mismatch**:
   - C4 is generic web text, not a proper control for ToM
   - Better approach: use ToM control conditions (correct label, informed protagonist)

3. **Limited Training Data**:
   - Only 100 samples may not capture full diversity of ToM scenarios
   - Consider using 1000+ samples for more stable estimates

### Recommendations

- See Notebook 2 for improved methodology using contrast pair datasets
- Consider per-token normalization: divide FIM by number of supervised tokens
- Use diverse ToM scenarios across all task types (S1, S2, S3)